# 📘 Week 1 · Day 3-4: NumPy & Pandas 실전

## 🎯 학습 목표
- NumPy의 벡터·행렬 연산 이해
- Pandas의 DataFrame·Series 자유롭게 조작
- 실제 Kaggle 데이터로 **읽기·필터링·그룹·결합** 연습
- Kaggle에서 바로 써먹는 **유용한 코딩 패턴 30가지**

> 💡 **팁**: 이 노트북은 치트시트 형태입니다. 외우지 말고 **코드 패턴이 눈에 익도록** 반복 실행하세요.

---

## 1. NumPy 기초

In [ ]:
import numpy as np
import pandas as pd

# 1-1. 배열 생성
a = np.array([1, 2, 3, 4, 5])
b = np.zeros((3, 3))       # 3x3 영행렬
c = np.ones((2, 4))        # 2x4 1행렬
d = np.arange(0, 10, 2)    # [0, 2, 4, 6, 8]
e = np.linspace(0, 1, 5)   # [0, 0.25, 0.5, 0.75, 1.0]
f = np.random.randn(3, 3)  # 표준정규 샘플

print("a:", a)
print("d:", d)
print("e:", e)
print("f shape:", f.shape)

In [ ]:
# 1-2. 벡터화 연산 (Python for문보다 100배 이상 빠름)
x = np.array([1, 2, 3, 4, 5])

print("x + 10   :", x + 10)
print("x * 2    :", x * 2)
print("x ** 2   :", x ** 2)
print("np.sqrt  :", np.sqrt(x))
print("np.log1p :", np.log1p(x))  # log(1+x) - Kaggle에서 자주 쓰는 변환
print("합, 평균, 표준편차:", x.sum(), x.mean(), x.std())

In [ ]:
# 1-3. Boolean 인덱싱 (Kaggle에서 가장 많이 쓰는 패턴)
scores = np.array([65, 82, 91, 45, 78, 99, 33, 67])
print("80 이상:", scores[scores >= 80])
print("평균보다 높은:", scores[scores > scores.mean()])

# 조건 여러 개
mask = (scores >= 60) & (scores < 90)
print("60~90:", scores[mask])

In [ ]:
# 1-4. np.where - 조건부 값 할당 (Feature Engineering 필수)
ages = np.array([15, 25, 35, 45, 65, 75])
age_group = np.where(ages < 20, 'teen',
             np.where(ages < 60, 'adult', 'senior'))
print(age_group)

---

## 2. Pandas 기초

### 2.1 Series와 DataFrame

In [ ]:
# Series: 1차원 (이름 있는 NumPy 배열)
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='value')
print(s)
print("s['b'] =", s['b'])

In [ ]:
# DataFrame: 2차원 테이블
df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Charlie', 'Dave', 'Eve'],
    'age':   [25, 32, 28, 45, 31],
    'score': [85, 72, 91, 58, 77],
    'city':  ['NY', 'LA', 'NY', 'SF', 'LA']
})
df

### 2.2 기본 탐색 (실전에서 가장 먼저 하는 것)

In [ ]:
# 형태 파악
print("shape   :", df.shape)
print("columns :", list(df.columns))
print("dtypes  :\n", df.dtypes)
print()
df.head(3)

In [ ]:
# 통계 요약 (수치형)
df.describe()

In [ ]:
# 범주형 포함 전체 요약
df.describe(include='all')

In [ ]:
# 결측치 확인
df.isna().sum()

### 2.3 선택 & 필터링

In [ ]:
# 열 선택
print(df['name'].tolist())
print()
print(df[['name', 'score']].head(2))

In [ ]:
# 행 선택 - loc (라벨), iloc (위치)
print("loc[0]   :", df.loc[0, 'name'])
print("iloc[0,0]:", df.iloc[0, 0])
print()
print(df.loc[1:3, ['name', 'score']])

In [ ]:
# 조건 필터링 (매우 자주 씀)
print(df[df['age'] > 30])
print()
print(df[(df['city'] == 'NY') & (df['score'] > 80)])

In [ ]:
# query 문법 (가독성 좋음)
df.query("age > 30 and city == 'LA'")

### 2.4 정렬

In [ ]:
# 단일 컬럼
df.sort_values('score', ascending=False)

In [ ]:
# 다중 컬럼
df.sort_values(['city', 'score'], ascending=[True, False])

### 2.5 그룹 & 집계 (pivot)

In [ ]:
# 기본 groupby
df.groupby('city')['score'].mean()

In [ ]:
# 여러 집계 함수 한 번에
df.groupby('city').agg(
    avg_score=('score', 'mean'),
    max_age=('age', 'max'),
    count=('name', 'count')
).reset_index()

In [ ]:
# pivot_table (엑셀의 피벗과 동일)
df2 = pd.DataFrame({
    'year': [2023, 2023, 2024, 2024, 2023, 2024],
    'city': ['NY', 'LA', 'NY', 'LA', 'NY', 'LA'],
    'sales':[100, 150, 120, 180, 110, 200]
})
df2.pivot_table(index='city', columns='year', values='sales', aggfunc='sum')

### 2.6 결합 (merge / concat)

In [ ]:
# concat: 세로로 쌓기
top = df.head(2)
bot = df.tail(2)
pd.concat([top, bot], ignore_index=True)

In [ ]:
# merge: SQL JOIN과 같음
users  = pd.DataFrame({'id': [1, 2, 3], 'name': ['A', 'B', 'C']})
orders = pd.DataFrame({'user_id': [1, 1, 2, 4], 'amount': [100, 50, 200, 300]})

# inner (교집합)
pd.merge(users, orders, left_on='id', right_on='user_id', how='inner')

In [ ]:
# left join (왼쪽 다 살림)
pd.merge(users, orders, left_on='id', right_on='user_id', how='left')

---

## 3. Kaggle 실전: Titanic 데이터로 EDA

데이터를 다운로드했다면 아래 경로를 맞춰주세요. 없으면 아래 셀에서 직접 생성합니다.

In [ ]:
# Titanic을 로컬에 받지 않았더라도 실습할 수 있게 샘플 데이터 생성
# 실제로는 pd.read_csv('../data/titanic/train.csv') 로 읽습니다
np.random.seed(42)
n = 891
titanic = pd.DataFrame({
    'PassengerId': range(1, n+1),
    'Survived':    np.random.binomial(1, 0.38, n),
    'Pclass':      np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
    'Sex':         np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
    'Age':         np.random.normal(30, 14, n).clip(0.5, 80),
    'SibSp':       np.random.poisson(0.5, n),
    'Parch':       np.random.poisson(0.38, n),
    'Fare':        np.random.exponential(32, n).round(2),
    'Embarked':    np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09])
})
# 결측치 의도적으로 만들기
titanic.loc[np.random.choice(n, 177, replace=False), 'Age'] = np.nan
titanic.loc[np.random.choice(n, 2, replace=False), 'Embarked'] = np.nan

print(titanic.shape)
titanic.head()

### 3.1 기본 정보

In [ ]:
titanic.info()

In [ ]:
titanic.describe(include='all')

### 3.2 결측치 탐색

In [ ]:
# 결측치 개수와 비율
missing = titanic.isna().sum()
missing_pct = (missing / len(titanic) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct(%)': missing_pct}).query('missing > 0')

### 3.3 타깃 변수(Survived) 분포

In [ ]:
print(titanic['Survived'].value_counts())
print()
print("생존률:", titanic['Survived'].mean().round(3))

### 3.4 그룹별 생존률 (가장 기본적인 EDA)

In [ ]:
# 성별 생존률
titanic.groupby('Sex')['Survived'].agg(['mean', 'count']).round(3)

In [ ]:
# 객실 등급별 생존률
titanic.groupby('Pclass')['Survived'].agg(['mean', 'count']).round(3)

In [ ]:
# 교차 분석 (성별 + 등급)
titanic.pivot_table(index='Pclass', columns='Sex', values='Survived', aggfunc='mean').round(3)

### 3.5 연령대별 생존률 (범주화)

In [ ]:
# pd.cut으로 구간 나누기
titanic['AgeGroup'] = pd.cut(
    titanic['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['child', 'teen', 'young', 'middle', 'senior']
)
titanic.groupby('AgeGroup', observed=False)['Survived'].mean().round(3)

---

## 4. Kaggle 필수 패턴 30가지 치트시트

In [ ]:
# 1. CSV 읽기 (대용량 최적화)
# pd.read_csv('train.csv', dtype={'id': 'int32', 'target': 'int8'})

# 2. 메모리 사용량 확인
print(titanic.memory_usage(deep=True).sum() / 1024**2, "MB")

In [ ]:
# 3. 메모리 절감 (int64 → int8/int16 등으로 다운캐스팅)
def reduce_memory(df):
    for col in df.select_dtypes(include='number').columns:
        col_min, col_max = df[col].min(), df[col].max()
        if pd.api.types.is_integer_dtype(df[col]):
            if col_min >= np.iinfo(np.int8).min and col_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        else:
            df[col] = pd.to_numeric(df[col], downcast='float')
    return df

before = titanic.memory_usage(deep=True).sum() / 1024**2
titanic_opt = reduce_memory(titanic.copy())
after = titanic_opt.memory_usage(deep=True).sum() / 1024**2
print(f"Before: {before:.2f} MB → After: {after:.2f} MB")

In [ ]:
# 4. 고유값 개수
titanic.nunique()

In [ ]:
# 5. 값 빈도
titanic['Embarked'].value_counts(dropna=False)

In [ ]:
# 6. 결측치 채우기 (간단 버전)
df_fill = titanic.copy()
df_fill['Age'] = df_fill['Age'].fillna(df_fill['Age'].median())
df_fill['Embarked'] = df_fill['Embarked'].fillna(df_fill['Embarked'].mode()[0])
df_fill.isna().sum()

In [ ]:
# 7. 그룹별 결측치 채우기 (더 정확)
df_fill2 = titanic.copy()
df_fill2['Age'] = df_fill2.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
df_fill2['Age'].isna().sum()

In [ ]:
# 8. apply vs map vs vectorize
# map: Series에만 사용
titanic['Sex_map'] = titanic['Sex'].map({'male': 0, 'female': 1})

# apply: 행/열 단위 함수 적용 (느림, 최후 수단)
titanic['fare_log'] = titanic['Fare'].apply(np.log1p)

# 가장 좋은 방법: 벡터화
titanic['fare_log2'] = np.log1p(titanic['Fare'])
titanic[['Sex', 'Sex_map', 'Fare', 'fare_log']].head(3)

In [ ]:
# 9. 원-핫 인코딩
pd.get_dummies(titanic[['Sex', 'Embarked']], drop_first=True).head(3)

In [ ]:
# 10. 문자열 처리
# 이름에서 경칭 추출 (Mr, Mrs, Miss)
titanic['FakeName'] = 'Mr. John Doe'  # 예시용
titanic['FakeName'].str.extract(r'(\w+)\.').head(3)

---

## 📝 Day 3-4 체크리스트
- [ ] NumPy 배열 연산·Boolean 인덱싱 이해
- [ ] Pandas DataFrame 선택/필터/정렬 자유자재
- [ ] groupby + agg 조합 사용 가능
- [ ] merge/concat 실습 완료
- [ ] Titanic 데이터로 기본 EDA 수행

다음 노트북에서는 **시각화(Matplotlib/Seaborn)** 와 **EDA 전략**을 배웁니다.